# moeforecast quickstart

This notebook builds a Mixture-of-Experts forecaster step by step on a synthetic hourly demand series. Replace the data cell with your own series; every later cell works unchanged.

In [ ]:
from moeforecast import (
    AdaptiveGating,
    ExpertSelector,
    FeatureBuilder,
    MoEForecaster,
    ProfileClusterer,
    SeriesAnalyzer,
    compare_classifiers,
    feature_relevance,
    prepare_series,
    temporal_train_test_split,
)
from moeforecast.datasets import make_demand_series
from moeforecast.metrics import mae, mape, mase

## 1. Data

`prepare_series` accepts a Series or a DataFrame (with `target` and `time_col`). It sorts the data, checks for duplicates, puts it on a regular grid and reports the gaps.

In [ ]:
raw = make_demand_series(n_days=2 * 364)  # replace with your own data
y, report = prepare_series(raw, return_report=True)
print(report)

train, test = temporal_train_test_split(y, test_size=24 * 28)  # last 4 weeks for testing
y.plot(figsize=(12, 3), linewidth=0.4, title="Hourly demand");

## 2. Which seasonal periods and lags matter?

`SeriesAnalyzer` reads the autocorrelation structure. It suggests periods from the ACF peaks and lags from the significant partial autocorrelations, plus each period and its neighbours. It is computed on the training part only.

In [ ]:
analysis = SeriesAnalyzer().fit(train)
print(analysis.summary())
analysis.plot_acf_pacf()
analysis.plot_seasonal_profiles();

## 3. Features and their relevance

`FeatureBuilder` creates causal features: the value at time *t* uses only data before *t*. Permutation importance on a later hold-out shows which features the model actually relies on.

In [ ]:
features = FeatureBuilder(lags=analysis.suggested_lags_, fourier_periods=analysis.suggested_periods_)
X = features.fit(train).transform(train)
relevance = feature_relevance(X, train, method="permutation")
relevance.head(10)

## 4. Regimes: day types and week types

Each day (24-hour profile) and each week (168-hour profile) is clustered. The number of clusters is chosen by the silhouette score unless `n_clusters` is given.

In [ ]:
days = ProfileClusterer("D", name="day", k_range=range(2, 7)).fit(train)
weeks = ProfileClusterer("W", name="week", n_clusters=2).fit(train)
print(f"day types: {days.n_clusters_}, week types: {weeks.n_clusters_}")
days.plot_scores()
days.plot_profiles();

## 5. Can the regimes be predicted from the features?

The gating needs classifiers that predict the regime of each hour. `compare_classifiers` scores candidate classifiers with time-ordered folds.

In [ ]:
compare_classifiers(X, days.labels_for(train.index), n_splits=3)

## 6. The Mixture of Experts

`MoEForecaster` fits everything above on the training series. It then selects one expert per (day type, week type) combination, using forward-chaining cross-validation.

In [ ]:
model = MoEForecaster(
    features=features,
    clusterers=[days, weeks],
    selector=ExpertSelector(metric="mae", n_splits=3),
    gating=AdaptiveGating(threshold=0.75),
).fit(train)

model.selector_.results_.query("selected")

## 7. Evaluation on the test period

The model was fitted on `train` only. Predicting on the full series gives one-step-ahead forecasts for the test period, each using only the observations before it. `predict_with_metadata` shows which expert was used and whether the gating was hard or soft.

In [ ]:
pred, meta = model.predict_with_metadata(y)
pred, meta = pred[test.index], meta.loc[test.index]

print(f"MAE  {mae(test, pred):.2f}")
print(f"MAPE {mape(test, pred):.2%}")
print(f"MASE {mase(test, pred, train, m=168):.3f}  (< 1 beats the weekly seasonal naive)")
print(meta["mode"].value_counts().to_string())

ax = test.iloc[: 24 * 7].plot(figsize=(12, 3), label="actual")
pred.iloc[: 24 * 7].plot(ax=ax, label="MoE one-step-ahead")
ax.legend();

## 8. Multi-step forecast and saving the model

`forecast` is recursive: each prediction is fed back as the latest observation. `save` and `load` keep the library version alongside the model.

In [ ]:
from moeforecast import load, save

week_ahead = model.forecast(train, steps=24 * 7)
print(f"MAE of the recursive 7-day forecast: {mae(test.iloc[: 24 * 7], week_ahead):.2f}")

save(model, "moe_model.joblib")
reloaded = load("moe_model.joblib")